# Evaluation: LLM Judge + Analysis

Loads the raw inference results from Notebook 04, scores all 180 responses with an LLM judge (Claude Haiku), and produces a comparison summary table.<br>
<br>
**Pipeline position:** 01 Fine-Tuning → 02 Data Generation → 03 Student Distillation → 04 Inference → `[05 Evaluation]`<br>
**No GPU required.** Runs locally — reads from `results/` and calls the Anthropic API.<br>
[![Open Kaggle Version](https://img.shields.io/badge/Open%20Kaggle%20Version-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white)](https://www.kaggle.com/code/dennisfeyerabend/05-evaluation-kaggle)

## 1. Setup

Install dependencies and import standard libraries.

In [1]:
%%capture
!pip install -r ../requirements.txt

In [2]:
import json
import os
from datetime import date
import anthropic

RESULTS_DIR = "../results"

print(f"anthropic:   {anthropic.__version__}")
print(f"Results dir: {os.path.abspath(RESULTS_DIR)}")

anthropic:   0.86.0
Results dir: C:\Users\Battlestation\PycharmProjects\00_python-ki-advanced\distil-support-llm\results


## 2. Authentication

Loads `ANTHROPIC_API_KEY` from a local `.env` file.<br>
Create a `.env` file in the repo root with the following line if you haven't already:<br>
`ANTHROPIC_API_KEY=your_key_here`

In [3]:
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

if ANTHROPIC_API_KEY:
    print(f"ANTHROPIC_API_KEY set. ({ANTHROPIC_API_KEY[:4]}...{ANTHROPIC_API_KEY[-4:]})")
else:
    print("WARNING: ANTHROPIC_API_KEY not set — add it to your .env file.")

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("Anthropic client ready.")

ANTHROPIC_API_KEY set. (sk-a...GgAA)
Anthropic client ready.


## 3. Load Data and Define Judge

Loads the four result files from `results/` and defines the LLM judge helper used in Section 4.

In [4]:
def load_json(filename):
    path = os.path.join(RESULTS_DIR, filename)
    with open(path, encoding="utf-8") as f:
        return json.load(f)

base_data    = load_json("raw_generations_base.json")
teacher_data = load_json("raw_generations_teacher.json")
student_data = load_json("raw_generations_student.json")
vram_data    = load_json("vram_measurements.json")

all_entries = (
    [{"model_label": "base",    **e} for e in base_data] +
    [{"model_label": "teacher", **e} for e in teacher_data] +
    [{"model_label": "student", **e} for e in student_data]
)

print(f"base:     {len(base_data)} entries")
print(f"teacher:  {len(teacher_data)} entries")
print(f"student:  {len(student_data)} entries")
print(f"combined: {len(all_entries)} entries")
print(f"vram:     {len(vram_data)} entries")

base:     60 entries
teacher:  60 entries
student:  60 entries
combined: 180 entries
vram:     3 entries


In [5]:
JUDGE_MODEL = "claude-haiku-4-5-20251001"

JUDGE_SYSTEM_PROMPT = """\
You are evaluating a customer support response written in German.
Score the response on 4 binary criteria. Each is 1 (met) or 0 (not met).

1. acknowledgement: Does the response open with empathy, recognition, or \
acknowledgement of the customer's situation (e.g., "Das tut mir leid", \
"Ich verstehe", "Vielen Dank für Ihre Anfrage")?
2. structured_steps: Does the response provide its main guidance as numbered \
steps, bullet points, or clearly enumerated items (NOT flowing prose)?
3. closing: Does the response end with an offer to help further, a follow-up \
invitation, or a clear professional closing?
4. tone: Is the overall tone professional, polite, and respectful (Sie-form, \
no rudeness, no overly casual language)?

Respond ONLY with valid JSON. No prose, no markdown:
{"acknowledgement": 0|1, "structured_steps": 0|1, "closing": 0|1, "tone": 0|1}"""


def call_judge(query_text, response_text):
    user_prompt = (
        f"Customer query (for context):\n{query_text}\n\n"
        f"Response to evaluate:\n{response_text}"
    )
    for attempt in range(2):
        message = client.messages.create(
            model=JUDGE_MODEL,
            max_tokens=64,
            temperature=0,                  # deterministic scoring
            system=JUDGE_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_prompt}],
        )
        raw = message.content[0].text.strip()

        if "```" in raw:
            raw = raw.replace("```json", "").replace("```", "").strip()

        try:
            scores = json.loads(raw)
            scores["judge_total"] = sum(
                scores[k] for k in ["acknowledgement", "structured_steps", "closing", "tone"]
            )
            return scores
        except json.JSONDecodeError:
            if attempt == 0:
                continue

    raise ValueError(f"Unparseable judge output after 2 attempts: {raw}")


print(f"Judge model: {JUDGE_MODEL}")
print("call_judge() ready.")

Judge model: claude-haiku-4-5-20251001
call_judge() ready.


## 4. Run LLM Judge

Scores all 180 responses across the three models using Claude Haiku as a blind judge.
Each call returns four binary scores (0 or 1): acknowledgement, structured steps, closing offer, and professional tone.

Run the sanity check cell first to spot any rubric issues before committing to 180 API calls.
Resumable: if the run is interrupted, re-running the sweep cell will skip already-scored entries.

In [6]:
import random

for entry in random.sample(all_entries, 3):
    scores = call_judge(entry["query_text"], entry["response_text"])
    print(f"[{entry['model_label']}] {entry['query_id']}")
    print(f"Query:    {entry['query_text'][:150]}...")
    print(f"Response: {entry['response_text']}")
    print(f"Scores:   {scores}")
    print()

[student] q02
Query:    Koennten Sie mir bitte mitteilen, ob beim Versand eine Geschenkverpackung mit Karte moeglich ist....
Response: Natürlich, das koennen Sie! Eine Geschenkverpackung mit Karte ist ganz moeglich. Sie koennen die Karte auch an die Empfängerin senden oder an einen anderen Empfänger setzen.
Scores:   {'acknowledgement': 0, 'structured_steps': 0, 'closing': 0, 'tone': 1, 'judge_total': 1}

[base] q17
Query:    Ich warte seit fuenf Tagen auf das Ruecksendeetikett, das mir per E-Mail zugeschickt werden sollte und nichts kommt an....
Response: Tut mir leid zu hören, dass Sie nicht mehr darauf gewartet haben. Ich würde gerne helfen, wenn Sie möchten weitere Informationen oder müssen eine Lösung finden. Bitte schreibe mich nach diesen Fällen mit allen notwendigen Details.
Scores:   {'acknowledgement': 1, 'structured_steps': 0, 'closing': 0, 'tone': 0, 'judge_total': 1}

[base] q35
Query:    Abo nur pausieren geht das...
Response: Tut mir leid, aber ich kann keine spezifische

In [7]:
JUDGE_SCORES_PATH = os.path.join(RESULTS_DIR, "judge_scores.json")

if os.path.exists(JUDGE_SCORES_PATH):
    with open(JUDGE_SCORES_PATH, encoding="utf-8") as f:
        judge_scores = json.load(f)
    scored = {(s["query_id"], s["model_label"]) for s in judge_scores}
    print(f"Loaded {len(judge_scores)} existing scores — {len(all_entries) - len(scored)} remaining.")
else:
    judge_scores = []
    scored = set()
    print(f"No existing scores — scoring all {len(all_entries)} entries.")

newly_scored = 0

for entry in all_entries:
    key = (entry["query_id"], entry["model_label"])
    if key in scored:
        continue

    scores = call_judge(entry["query_text"], entry["response_text"])

    judge_scores.append({
        "query_id":         entry["query_id"],
        "model_label":      entry["model_label"],
        "acknowledgement":  scores["acknowledgement"],
        "structured_steps": scores["structured_steps"],
        "closing":          scores["closing"],
        "tone":             scores["tone"],
        "judge_total":      scores["judge_total"],
    })
    scored.add(key)
    newly_scored += 1

    if newly_scored % 10 == 0:
        print(f"  scored {len(judge_scores)}/{len(all_entries)}")

with open(JUDGE_SCORES_PATH, "w", encoding="utf-8") as f:
    json.dump(judge_scores, f, ensure_ascii=False, indent=2)

print(f"\nDone. {newly_scored} new entries scored.")
print(f"Saved: {JUDGE_SCORES_PATH}  ({len(judge_scores)} total entries)")

Loaded 180 existing scores — 0 remaining.

Done. 0 new entries scored.
Saved: ../results\judge_scores.json  (180 total entries)


## 5. Word Count Analysis

Distribution of response lengths per model, measured in words.
Word counts were recorded during inference in Notebook 04.

In [8]:
model_data = {
    "base":    base_data,
    "teacher": teacher_data,
    "student": student_data,
}

print("Word count per model\n")
print(f"{'Model':<10} {'Mean':>6} {'Median':>8} {'Min':>6} {'Max':>6}")
print("-" * 40)

for label, data in model_data.items():
    counts = sorted(e["word_count"] for e in data)
    n      = len(counts)
    mean   = sum(counts) / n
    median = counts[n // 2]
    print(f"{label:<10} {mean:>6.1f} {median:>8} {counts[0]:>6} {counts[-1]:>6}")

Word count per model

Model        Mean   Median    Min    Max
----------------------------------------
base         70.4       53     14    223
teacher      38.1       38      9     93
student      36.5       35      9     79


## 6. Tokens/sec Analysis

Inference speed per model, measured in tokens per second.

Wall-clock time per query is not used here because it is confounded by output length — a model that generates more tokens will always appear slower even at identical speed.
Tokens/sec measures pure inference throughput and is independent of response length.

In [9]:
print("Tokens/sec per model\n")
print(f"{'Model':<10} {'Mean':>7} {'Median':>8} {'Min':>7} {'Max':>7}")
print("-" * 43)

for label, data in model_data.items():
    speeds = sorted(e["tokens_per_sec"] for e in data)
    n      = len(speeds)
    mean   = sum(speeds) / n
    median = speeds[n // 2]
    print(f"{label:<10} {mean:>7.1f} {median:>8.1f} {speeds[0]:>7.1f} {speeds[-1]:>7.1f}")

Tokens/sec per model

Model         Mean   Median     Min     Max
-------------------------------------------
base          23.7     24.0    21.4    24.8
teacher       12.9     12.8    12.3    13.5
student       10.5     10.6     9.6    11.0


## 7. VRAM Analysis

Peak VRAM usage per model recorded during inference in Notebook 04.
All three models were loaded with `load_in_4bit=True` for a fair comparison.

In [10]:
print("Peak VRAM per model\n")

max_vram = max(e["vram_peak_gb"] for e in vram_data)
bar_width = 28

for entry in vram_data:
    label  = entry["model_label"]
    gb     = entry["vram_peak_gb"]
    filled = round((gb / max_vram) * bar_width)
    bar    = "█" * filled + "░" * (bar_width - filled)
    print(f"{label:<10} {bar}  {gb:.2f} GB")

Peak VRAM per model

base       ███████████████████░░░░░░░░░  1.63 GB
teacher    ████████████████████████████  2.36 GB
student    ███████████████░░░░░░░░░░░░░  1.24 GB


## 8. Why Not BERTScore?

BERTScore measures semantic similarity between a generated response and a reference text.
It answers the question: *does this response say the same thing as the reference?*

That is not what this project trains for. The teacher model was fine-tuned for **style and format transfer** — structured steps, polite tone, acknowledgement of the customer's situation, a closing offer to help.
These are structural and tonal properties, not semantic content.

A base model that writes a perfectly correct but unstructured prose response would score high on BERTScore and low on every criterion this project actually cares about. Including BERTScore would obscure the evaluation rather than clarify it.

The LLM judge in Section 4 scores exactly the properties the training targeted.
Word count and tokens/sec measure the practical trade-offs of distillation.
Together these three cover what matters for this project.

## 9. Summary Table

Aggregated results across all three models. Percentages show the share of responses (out of 60) that met each criterion.

In [11]:
vram_by_model = {e["model_label"]: e["vram_peak_gb"] for e in vram_data}

summary = []
for label in ["base", "teacher", "student"]:
    scores = [s for s in judge_scores if s["model_label"] == label]
    data   = model_data[label]
    n      = len(scores)

    summary.append({
        "model_label":          label,
        "format_score_mean":    round(sum(s["judge_total"]      for s in scores) / n, 2),
        "acknowledgement_pct":  round(sum(s["acknowledgement"]  for s in scores) / n * 100, 1),
        "structured_steps_pct": round(sum(s["structured_steps"] for s in scores) / n * 100, 1),
        "closing_pct":          round(sum(s["closing"]          for s in scores) / n * 100, 1),
        "tone_pct":             round(sum(s["tone"]             for s in scores) / n * 100, 1),
        "word_count_mean":      round(sum(e["word_count"]       for e in data)   / len(data), 1),
        "tokens_per_sec_mean":  round(sum(e["tokens_per_sec"]   for e in data)   / len(data), 1),
        "vram_peak_gb":         vram_by_model[label],
    })

b, t, s = summary[0], summary[1], summary[2]
print("Metrics computed.")

Metrics computed.


In [12]:
col = 24
print(f"{'Metric':<26} {'Base (1.5B)':>{col}} {'Teacher (3B+LoRA)':>{col}} {'Student (1.5B)':>{col}}")
print("-" * (26 + col * 3))

rows = [
    ("Format Score (0–4)",  f"{b['format_score_mean']:.2f}",    f"{t['format_score_mean']:.2f}",    f"{s['format_score_mean']:.2f}"),
    ("  Acknowledgement",   f"{b['acknowledgement_pct']:.1f}%", f"{t['acknowledgement_pct']:.1f}%", f"{s['acknowledgement_pct']:.1f}%"),
    ("  Structured Steps",  f"{b['structured_steps_pct']:.1f}%",f"{t['structured_steps_pct']:.1f}%",f"{s['structured_steps_pct']:.1f}%"),
    ("  Closing",           f"{b['closing_pct']:.1f}%",         f"{t['closing_pct']:.1f}%",         f"{s['closing_pct']:.1f}%"),
    ("  Professional Tone", f"{b['tone_pct']:.1f}%",            f"{t['tone_pct']:.1f}%",            f"{s['tone_pct']:.1f}%"),
    ("Avg Word Count",      f"{b['word_count_mean']:.1f}",      f"{t['word_count_mean']:.1f}",      f"{s['word_count_mean']:.1f}"),
    ("Avg Tokens/sec",      f"{b['tokens_per_sec_mean']:.1f}",  f"{t['tokens_per_sec_mean']:.1f}",  f"{s['tokens_per_sec_mean']:.1f}"),
    ("Peak VRAM (GB)",      f"{b['vram_peak_gb']:.2f}",         f"{t['vram_peak_gb']:.2f}",         f"{s['vram_peak_gb']:.2f}"),
]

for metric, bv, tv, sv in rows:
    print(f"{metric:<26} {bv:>{col}} {tv:>{col}} {sv:>{col}}")

# Save eval_summary.json
with open(os.path.join(RESULTS_DIR, "eval_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# Save eval_metadata.json
metadata = {
    "judge_model":       JUDGE_MODEL,
    "judge_temperature": 0,
    "evaluation_date":   date.today().isoformat(),
    "torch_seed_base":   42,
    "generation_params": {
        "temperature":    0.7,
        "top_p":          0.9,
        "max_new_tokens": 384,
        "do_sample":      True,
    },
    "n_queries": 60,
}
with open(os.path.join(RESULTS_DIR, "eval_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("\nSaved: eval_summary.json")
print("Saved: eval_metadata.json")

Metric                                  Base (1.5B)        Teacher (3B+LoRA)           Student (1.5B)
--------------------------------------------------------------------------------------------------
Format Score (0–4)                             1.73                     1.63                     1.80
  Acknowledgement                             55.0%                    41.7%                    50.0%
  Structured Steps                            26.7%                    53.3%                    50.0%
  Closing                                     56.7%                    11.7%                    18.3%
  Professional Tone                           35.0%                    56.7%                    61.7%
Avg Word Count                                 70.4                     38.1                     36.5
Avg Tokens/sec                                 23.7                     12.9                     10.5
Peak VRAM (GB)                                 1.63                     2.36         